# 01 — EDA (FathomNetCLEF2026)

**Inputs:** output of `00_setup_download_data` (contains `train_dataset.json`,
`test_dataset.json`, `images/`).

Class distribution, bbox statistics, and sample visualizations —
material for the Dataset section of your working note / research paper.

In [ ]:
from pathlib import Path
import json, glob

# Works on Kaggle (00's saved output as Input) or locally
hits = glob.glob('/kaggle/input/**/train_dataset.json', recursive=True)
ANN = Path(hits[0]).parent if hits else Path('/kaggle/working')
IMG = ANN.parent / 'images'
if not IMG.exists():
    IMG = ANN / 'images'
print('Annotations:', ANN, '\nImages dir:', IMG)

train = json.load(open(ANN / 'train_dataset.json'))
test = json.load(open(ANN / 'test_dataset.json'))
print(f"train: {len(train['images'])} imgs, {len(train['annotations'])} anns, "
      f"{len(train['categories'])} cats")
print(f"test : {len(test['images'])} imgs, {len(test.get('annotations', []))} anns")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cats = {c['id']: c['name'] for c in train['categories']}
df = pd.DataFrame(train['annotations'])
df['name'] = df['category_id'].map(cats)

counts = df['name'].value_counts()
plt.figure(figsize=(8, 10))
sns.barplot(x=counts.values, y=counts.index, color='#3a7ca5')
plt.xscale('log')
plt.title('Annotation counts per class (train) — long-tail is expected')
plt.xlabel('boxes')
plt.tight_layout(); plt.show()

b = np.array(df['bbox'].tolist())
print('bbox stats (w, h, area):')
print(pd.DataFrame({'w': b[:, 2], 'h': b[:, 3], 'area': b[:, 2] * b[:, 3]}).describe())

In [ ]:
# Visualize annotated samples (paper figures)
import matplotlib.patches as patches
from PIL import Image

anns_by_img = {}
for a in train['annotations']:
    anns_by_img.setdefault(a['image_id'], []).append(a)

have = [i for i in train['images'] if (IMG / i['file_name']).exists()][:8]
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, im in zip(axes.ravel(), have):
    ax.imshow(Image.open(IMG / im['file_name']))
    for a in anns_by_img.get(im['id'], []):
        x, y, w, h = a['bbox']
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, ec='red', lw=1.5))
        ax.text(x, max(y - 4, 8), cats[a['category_id']], color='yellow', fontsize=7,
                bbox={'fc': 'black', 'alpha': 0.6, 'pad': 1})
    ax.axis('off')
plt.tight_layout(); plt.show()